# Mini-Epoch Demo: Audio Dataset with `nb_samples_per_epoch`

This notebook demonstrates the **mini-epoch** feature of `VVTKDataLoader`.

**Setup:**
- 1 000 synthetic audio samples (16 kHz, 1–5 s) + token sequences
- `nb_samples_per_epoch=100` → 10 batches of 10 per mini-epoch
- 20 mini-epochs → the full dataset (1 000 samples) is traversed **twice**
- Reshuffle happens after mini-epoch 10 (all 1 000 samples consumed), then again after 20

We compare:
1. **VVTKDataset + PyTorch DataLoader** — manual mini-epoch via `Subset` / index slicing
2. **VVTKDataset + VVTKDataLoader** — built-in `nb_samples_per_epoch` support

In [1]:
import torch
import numpy as np
import os, time
from vvtk_dataset import VVTKDataset, VVTKDataLoader

## Step 1 — Generate 1 000 synthetic audio + token samples and store in VVTK

In [2]:
N = 1_000
SR = 16_000
MIN_DUR, MAX_DUR = 1, 5
MAX_AUDIO_SAMPLES = SR * MAX_DUR   # 80 000
MAX_TOKEN_LEN = 100
VVTK_PATH = 'data/shards/train'

os.makedirs(os.path.dirname(VVTK_PATH), exist_ok=True)
rng = np.random.default_rng(42)

print(f'Generating {N} samples and writing VVTK shards ...')
t0 = time.time()

with VVTKDataset(VVTK_PATH, mode='wb', num_shards=8,
                 compression=['flac', 'none'],
                 compression_args=[{'sample_rate': SR}, {}]) as ds:
    for i in range(N):
        dur = rng.uniform(MIN_DUR, MAX_DUR)
        n_samp = int(SR * dur)
        freq = rng.uniform(200, 4000)
        t = np.arange(n_samp, dtype=np.float32) / SR
        waveform = (0.5 * np.sin(2 * np.pi * freq * t)).astype(np.float32)

        tok_len = int(10 + 90 * (dur - MIN_DUR) / (MAX_DUR - MIN_DUR))
        tokens = rng.integers(0, 30000, size=tok_len, dtype=np.int16)

        ds.add(i, waveform, tokens)

print(f'Done in {time.time()-t0:.1f}s — {N} samples in 8 shards')

Generating 1000 samples and writing VVTK shards ...
[VVTK] Saved dataset to data/shards/train
Done in 1.0s — 1000 samples in 8 shards


## Step 2 — VVTKDataset + PyTorch DataLoader (manual mini-epochs)

PyTorch's `DataLoader` reshuffles on every `__iter__` call, so to get the same
"consume-all-then-reshuffle" behaviour we manage a shuffled index list manually
and slice it into mini-epoch chunks.

In [3]:
BATCH_SIZE = 10
NB_SAMPLES_PER_EPOCH = 100   # 10 batches per mini-epoch
NUM_MINI_EPOCHS = 20

ds = VVTKDataset(VVTK_PATH, mode='rb',
                 compression=['flac', 'none'],
                 compression_args=[{'sample_rate': SR}, {}])

print(f'Dataset size: {len(ds)}')
print(f'Mini-epoch size: {NB_SAMPLES_PER_EPOCH} samples  ({NB_SAMPLES_PER_EPOCH // BATCH_SIZE} batches)')
print(f'Full passes needed: {len(ds) // NB_SAMPLES_PER_EPOCH} mini-epochs')
print()

# --- Manual mini-epoch logic for PyTorch DataLoader ---
indices = None
offset = 0

for epoch in range(1, NUM_MINI_EPOCHS + 1):
    # Reshuffle when the full dataset has been consumed (or first time)
    if indices is None or offset >= len(ds):
        indices = torch.randperm(len(ds)).tolist()
        offset = 0
        print(f'  [Reshuffle] at mini-epoch {epoch}')

    # Slice the indices for this mini-epoch
    end = min(offset + NB_SAMPLES_PER_EPOCH, len(ds))
    subset = torch.utils.data.Subset(ds, indices[offset:end])
    loader = torch.utils.data.DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False)
    offset = end

    t0 = time.time()
    batches = 0
    samples = 0
    for batch in loader:
        # Each item in batch is a tensor (no lengths with PyTorch DL on VVTKDataset without fixed_shapes)
        batches += 1
        samples += batch[0].shape[0]
    elapsed = time.time() - t0
    print(f'  Mini-epoch {epoch:2d}: {batches:3d} batches, {samples:4d} samples, {elapsed:.3f}s')

print(f'\nTotal: {NUM_MINI_EPOCHS} mini-epochs = {NUM_MINI_EPOCHS * NB_SAMPLES_PER_EPOCH} sample-reads '
      f'({NUM_MINI_EPOCHS * NB_SAMPLES_PER_EPOCH / len(ds):.0f}x full passes)')

Dataset size: 1000
Mini-epoch size: 100 samples  (10 batches)
Full passes needed: 10 mini-epochs

  [Reshuffle] at mini-epoch 1


RuntimeError: stack expects each tensor to be equal size, but got [59679] at entry 0 and [36160] at entry 1

## Step 3 — VVTKDataset + VVTKDataLoader with `nb_samples_per_epoch`

The C++ loader handles everything: mini-epoch boundaries, sample counting,
and delayed reshuffling — no manual index management needed.

In [ ]:
vvtk_ds = VVTKDataset(VVTK_PATH, mode='rb',
                      compression=['flac', 'none'],
                      compression_args=[{'sample_rate': SR}, {}])

vvtk_loader = VVTKDataLoader(
    vvtk_ds,
    batch_size=BATCH_SIZE,
    num_workers=4,
    ring_size=4,
    shapes=[(MAX_AUDIO_SAMPLES,), (MAX_TOKEN_LEN,)],
    dtypes=[torch.float32, torch.int16],
    padding_values=[0.0, 0],
    shuffle=True,
    nb_samples_per_epoch=NB_SAMPLES_PER_EPOCH   # <-- the only addition
)

print(f'VVTKDataLoader: {len(vvtk_loader)} batches/mini-epoch (batch_size={BATCH_SIZE})')
print(f'  Audio shape: ({MAX_AUDIO_SAMPLES},) float32  |  Tokens: ({MAX_TOKEN_LEN},) int16')
print(f'  Full dataset: {len(vvtk_ds)} samples → reshuffle every {len(vvtk_ds) // NB_SAMPLES_PER_EPOCH} mini-epochs')
print()

for epoch in range(1, NUM_MINI_EPOCHS + 1):
    t0 = time.time()
    batches = 0
    samples = 0
    for batch in vvtk_loader:
        (audio, audio_lengths), (tokens, token_lengths) = batch
        batches += 1
        samples += audio.shape[0]
    elapsed = time.time() - t0
    print(f'  Mini-epoch {epoch:2d}: {batches:3d} batches, {samples:4d} samples, {elapsed:.3f}s')

print(f'\nTotal: {NUM_MINI_EPOCHS} mini-epochs = {NUM_MINI_EPOCHS * NB_SAMPLES_PER_EPOCH} sample-reads '
      f'({NUM_MINI_EPOCHS * NB_SAMPLES_PER_EPOCH / len(vvtk_ds):.0f}x full passes)')

## Step 4 — Resuming from a checkpoint with `set_mini_epoch`

When `shuffle=False` and data is consumed sequentially, a killed process can
resume from the exact mini-epoch it was at.  Save the mini-epoch counter in
your checkpoint, then call `loader.set_mini_epoch(n)` before iterating.

The method resets the C++ core and fast-forwards through the required batches
so the next `for batch in loader` picks up exactly where training left off.

In [ ]:
# --- Simulate training that gets interrupted at mini-epoch 7 ---

resume_ds = VVTKDataset(VVTK_PATH, mode='rb',
                        compression=['flac', 'none'],
                        compression_args=[{'sample_rate': SR}, {}])

resume_loader = VVTKDataLoader(
    resume_ds,
    batch_size=BATCH_SIZE,
    num_workers=4,
    ring_size=4,
    shapes=[(MAX_AUDIO_SAMPLES,), (MAX_TOKEN_LEN,)],
    dtypes=[torch.float32, torch.int16],
    padding_values=[0.0, 0],
    shuffle=False,                              # sequential — deterministic order
    nb_samples_per_epoch=NB_SAMPLES_PER_EPOCH
)

# Phase 1: train normally for 7 mini-epochs, then "crash"
CRASH_AT = 7
print(f'--- Phase 1: training mini-epochs 0..{CRASH_AT - 1}, then simulated crash ---')
for epoch in range(CRASH_AT):
    batches = 0
    for batch in resume_loader:
        (audio, audio_lengths), (tokens, token_lengths) = batch
        batches += 1
    print(f'  Mini-epoch {epoch:2d}: {batches} batches')

# (save checkpoint here in real code: torch.save({..., 'mini_epoch': CRASH_AT}, path))
print(f'\n  ** Process killed at mini-epoch {CRASH_AT} **')

# Phase 2: new process — restore checkpoint, jump to mini-epoch 7
print(f'\n--- Phase 2: resume from checkpoint at mini-epoch {CRASH_AT} ---')

resume_ds2 = VVTKDataset(VVTK_PATH, mode='rb',
                         compression=['flac', 'none'],
                         compression_args=[{'sample_rate': SR}, {}])

resume_loader2 = VVTKDataLoader(
    resume_ds2,
    batch_size=BATCH_SIZE,
    num_workers=4,
    ring_size=4,
    shapes=[(MAX_AUDIO_SAMPLES,), (MAX_TOKEN_LEN,)],
    dtypes=[torch.float32, torch.int16],
    padding_values=[0.0, 0],
    shuffle=False,
    nb_samples_per_epoch=NB_SAMPLES_PER_EPOCH
)

resume_loader2.set_mini_epoch(CRASH_AT)     # <-- fast-forward to mini-epoch 7

for epoch in range(CRASH_AT, NUM_MINI_EPOCHS):
    batches = 0
    for batch in resume_loader2:
        (audio, audio_lengths), (tokens, token_lengths) = batch
        batches += 1
    print(f'  Mini-epoch {epoch:2d}: {batches} batches')

print(f'\nDone — mini-epochs {CRASH_AT}..{NUM_MINI_EPOCHS - 1} completed after resume.')

## Summary

| Approach | Mini-epoch code | Reshuffle control | Resume support |
|----------|----------------|-------------------|----------------|
| PyTorch DataLoader | Manual index slicing + `Subset` | Manual — track offset, regenerate permutation | Manual — save/restore offset + permutation |
| VVTKDataLoader | `nb_samples_per_epoch=100` | Automatic — reshuffles only after full dataset consumed | `set_mini_epoch(n)` — fast-forward to exact position |

With `nb_samples_per_epoch`:
- **Mini-epochs 1–10**: consume the 1 000 shuffled samples in order (100 per mini-epoch)
- **Mini-epoch 11**: automatic reshuffle, start second pass
- **Mini-epochs 11–20**: consume the newly-shuffled 1 000 samples

With `set_mini_epoch(n)`:
- Resets the C++ core and fast-forwards through `n * batches_per_mini_epoch` batches
- For `shuffle=False` (sequential): data order is **identical** to an uninterrupted run
- Save the mini-epoch counter in your checkpoint, then resume exactly where you left off
- For `shuffle=True`: each reset re-shuffles, so exact reproducibility requires external seed control